# Transfer Learning e Fine-Tuning com EfficientNetB1 — sintomas em folhas de café (BRACOL)

**Disciplina:** Inteligência Computacional / Machine Learning — Profa. Polyana Santos Fonseca Nascimento (CESUPA)
**Família / vaga:** EfficientNet — vaga 2 (a vaga 1 usa EfficientNetB0; nós usamos a **EfficientNetB1**)

**Objetivo:** classificar 5 sintomas em folhas de café arábica (saudável, bicho-mineiro, ferrugem, mancha de phoma e cercosporiose) com transfer learning, comparando a base congelada com duas estratégias de fine-tuning (A e B), com análise de erros e Grad-CAM.

**Como rodar:** Google Colab → *Ambiente de execução → Alterar tipo → GPU T4* → *Executar tudo*. A célula de configuração monta o Google Drive para salvar a divisão dos dados, os pesos e as figuras. O conjunto de teste só é liberado com `RUN_FINAL_TEST = True`, depois de registrada a escolha do modelo (Seções 11 e 12).

## 0. Integrantes e declaração de uso de IA Generativa

**Integrantes:** Vithor dos Santos, Murilo Pantoja e João Felipe

**Declaração de uso de IA Generativa**
1. **Ferramenta utilizada:** Claude Code (Anthropic), modelo Claude Opus 5.5.
2. **Finalidade do uso:** apoio na escolha do dataset (notebook-piloto comparando BRACOL, PBC e DeepWeeds), inspeção da integridade dos arquivos públicos do dataset e geração da estrutura e do código deste notebook.
3. **Extensão aproximada da contribuição:** o código do notebook foi gerado com IA e revisado pela equipe. A execução no Colab foi conduzida com apoio da IA, e os textos de interpretação (Seções 2 a 14) foram redigidos com apoio da IA a partir das saídas reais do próprio notebook e revisados pela equipe. A escolha do dataset e da arquitetura, as decisões finais e a análise dos resultados de treino são responsabilidade da equipe _(ajustar se a divisão do trabalho for diferente)_.
4. **Medidas adotadas para validação:** execução completa do notebook no Colab; conferência das contagens com o repositório dos autores; conferência dos números da arquitetura no modelo Keras real e no artigo original (Tan & Le, 2019); inspeção visual das duplicatas, dos erros e dos mapas Grad-CAM; leitura e discussão de cada seção pela equipe.

---
## 1. Ambiente, seeds e configuração

**Decisões:**
- **TensorFlow/Keras** (já vem no Colab), com `keras.applications.EfficientNetB1` e pesos oficiais do ImageNet.
- **Seeds fixas** em Python, NumPy e TensorFlow. Não usamos `enable_op_determinism`, porque algumas convoluções na GPU não têm versão determinística. Pode haver uma variação residual pequena entre execuções, e é por isso que reportamos intervalos de confiança.
- Todas as versões ficam registradas na tela e num JSON salvo junto com o modelo.

In [ ]:
!pip -q install imagehash

In [ ]:
import os, sys, io, re, gc, json, time, random, shutil, hashlib, zipfile, struct, subprocess, platform, warnings
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import PIL
from PIL import Image
import imagehash
import requests
import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix,
                             classification_report)
import tensorflow as tf
import keras
from keras import layers
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

VERSIONS = {
    "python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
    "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__, "Pillow": PIL.__version__, "imagehash": imagehash.__version__,
}
try:
    gpu_name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                              capture_output=True, text=True).stdout.strip()
except Exception:
    gpu_name = ""
VERSIONS["GPU"] = gpu_name or ("detectada" if tf.config.list_physical_devices("GPU")
                               else "NENHUMA — ative a T4 em Ambiente de execução > Alterar tipo")
display(pd.Series(VERSIONS, name="versão").to_frame())

In [ ]:
# ---------------- Configuração ----------------
IMG_SIZE = 240              # resolução nativa da EfficientNetB1
BATCH = 32
MAX_EPOCHS_FROZEN = 20      # teto; o early stopping decide quando parar
LR_FROZEN = 1e-3
PATIENCE = 4                # épocas sem melhora do F1 macro de validação
DROPOUT = 0.3
USE_AUGMENTATION = True     # só no treino
USE_CLASS_WEIGHTS = True    # compensa o desbalanceamento (ferrugem 991 x saudável 256)
N_BOOT = 1000               # bootstrap para intervalos de confiança
HASH_THR, NCC_THR = 6, 0.92 # detecção de quase-duplicatas (pHash + correlação de miniaturas)

RUN_FINE_TUNING = True      # executa as estratégias A e B (Seção 7)
RUN_FINAL_TEST = False      # True só depois de registrar a escolha do modelo (Seções 11 e 12)
FT_EPOCH_SCALE = 1.0        # multiplica as épocas de A e B (1.0 = plano original)
AUDIT_DOI = True            # baixa o zip do DOI só para documentar que está truncado
BRACOL_MANUAL_DIR = None    # pasta 'symptom' (com train/val/test/<classe>/ ou <classe>/), se o git falhar

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/cnn_efficientnetb1_bracol")
else:
    OUT_DIR = Path("saida_defesa").resolve()
DATA_DIR = Path("/content/data") if IN_COLAB else Path("data").resolve()
for d_ in [OUT_DIR, OUT_DIR / "figs", OUT_DIR / "modelos", DATA_DIR]:
    d_.mkdir(parents=True, exist_ok=True)

CONFIG = {k: v for k, v in globals().items()
          if k in ["SEED", "IMG_SIZE", "BATCH", "MAX_EPOCHS_FROZEN", "LR_FROZEN", "PATIENCE", "DROPOUT",
                   "USE_AUGMENTATION", "USE_CLASS_WEIGHTS", "N_BOOT", "HASH_THR", "NCC_THR"]}
with open(OUT_DIR / "config_versions.json", "w") as f:
    json.dump({"versions": VERSIONS, "config": CONFIG}, f, indent=2)
print("Saídas em:", OUT_DIR)

---
## 2. Arquitetura: EfficientNetB1

| Item | Resposta |
|---|---|
| **Arquitetura e versão** | EfficientNet, versão **B1** |
| **Autores e ano** | Mingxing Tan e Quoc V. Le (Google Brain), 2019 |
| **Referência** | TAN, M.; LE, Q. V. *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks*. In: International Conference on Machine Learning (ICML), PMLR 97, p. 6105–6114, 2019. arXiv:1905.11946 |
| **Implementação usada** | `keras.applications.EfficientNetB1` (TensorFlow/Keras), pesos ImageNet |

### 2.1 Ideia central e principal inovação

Numa CNN convencional, para ganhar acurácia aumenta-se **uma** dimensão por vez, e de forma arbitrária: mais camadas (ResNet-18 → ResNet-152), mais canais (Wide ResNet) ou imagens maiores. Os autores mostraram que o ganho satura rápido quando só uma dimensão cresce.

A EfficientNet propõe o **escalonamento composto (*compound scaling*)**: profundidade *d*, largura *w* e resolução *r* crescem **juntas**, controladas por um único coeficiente φ:

$$d = \alpha^{\phi},\quad w = \beta^{\phi},\quad r = \gamma^{\phi},\qquad \alpha\cdot\beta^{2}\cdot\gamma^{2} \approx 2$$

com α = 1,2, β = 1,1 e γ = 1,15, encontrados por busca em grade. Com a restrição, cada incremento de φ aproximadamente **dobra o custo (FLOPS)**. Intuição: imagens maiores precisam de mais camadas (campo receptivo maior) e de mais canais (padrões mais finos).

A rede-base **B0** não foi desenhada à mão: ela veio de uma **busca de arquitetura neural (NAS)** que otimiza acurácia e FLOPS ao mesmo tempo, no mesmo espaço de busca da MnasNet. As versões **B1 a B7** são a B0 escalonada.

### 2.2 Blocos

- **Stem:** convolução 3×3 com stride 2 (32 filtros) + *BatchNorm* + Swish.
- **MBConv (*Mobile inverted bottleneck*)**, herdado da MobileNetV2, é o bloco que se repete:
  1. **Expansão** 1×1, que multiplica os canais por 6 (exceto no estágio 1);
  2. **Convolução *depthwise*** k×k (k = 3 ou 5): um filtro por canal, muito mais barato que uma convolução comum;
  3. ***Squeeze-and-Excitation* (SE)**: um *pooling* global gera um vetor que passa por duas camadas densas e **repondera os canais**, uma "atenção" por canal;
  4. **Projeção** 1×1 linear (sem ativação), que volta a poucos canais;
  5. **Conexão residual** (soma) quando entrada e saída têm o mesmo formato, com *stochastic depth* (*drop connect*) no treino.
- **Ativação Swish/SiLU** (x·sigmoid(x)) no lugar da ReLU.
- **Topo:** convolução 1×1 para 1.280 canais → *Global Average Pooling* → classificador. No nosso caso, o classificador ImageNet de 1.000 classes é **substituído** por Dropout + Dense de 5 classes.
- **Na implementação Keras**, a normalização da entrada (*Rescaling* + *Normalization*) **já está dentro do modelo**: ele recebe pixels de 0 a 255.

### 2.3 Diferencial da versão B1 em relação à B0

| | B0 | **B1** |
|---|---|---|
| Coeficiente de profundidade / largura | 1,0 / 1,0 | **1,1 / 1,0** |
| Resolução de entrada | 224 | **240** |
| Blocos MBConv | 16 | **23** |
| Parâmetros (com o topo ImageNet) | 5,3 M | **7,8 M** |
| FLOPS | 0,39 B | **0,70 B** |
| Top-1 ImageNet (artigo) | 77,1% | **79,1%** |

A B1 é a B0 com **~10% mais profundidade por estágio** e **entrada 240×240**. Ganha 2 pontos no ImageNet e continua leve o suficiente para treinar e fazer fine-tuning na T4 do Colab gratuito.

A célula abaixo desenha o diagrama e **confere esses números no modelo Keras real** (blocos por estágio, formato de saída e parâmetros).

In [ ]:
B1_STAGES = [  # (rótulo, operador, kernel, canais de saída, repetições, resolução de saída com entrada 240)
    ("Stem", "Conv 3x3, s2", 3, 32, 1, "120x120"),
    ("Estágio 1", "MBConv1", 3, 16, 2, "120x120"),
    ("Estágio 2", "MBConv6", 3, 24, 3, "60x60"),
    ("Estágio 3", "MBConv6", 5, 40, 3, "30x30"),
    ("Estágio 4", "MBConv6", 3, 80, 4, "15x15"),
    ("Estágio 5", "MBConv6", 5, 112, 4, "15x15"),
    ("Estágio 6", "MBConv6", 5, 192, 5, "8x8"),
    ("Estágio 7", "MBConv6", 3, 320, 2, "8x8"),
    ("Topo", "Conv 1x1", 1, 1280, 1, "8x8"),
    ("Classificador (nosso)", "GAP + Dropout + Dense", "-", 5, 1, "5 classes"),
]

def draw_architecture():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 8.5), gridspec_kw={"width_ratios": [1.25, 1]})
    for a in (a1, a2):
        a.set_xlim(0, 10); a.set_ylim(0, 12); a.axis("off")
    a1.set_title("EfficientNetB1 (entrada 240x240x3)", fontsize=12, weight="bold")
    colors = {"Stem": "#d9d9d9", "Topo": "#d9d9d9", "Classificador (nosso)": "#f4b183"}
    y = 11.2
    for lab, op, k, ch, rep, res in B1_STAGES:
        box = FancyBboxPatch((0.6, y - 0.42), 8.8, 0.8, boxstyle="round,pad=0.05",
                             fc=colors.get(lab, "#9dc3e6"), ec="black")
        a1.add_patch(box)
        ktxt = f"k{k}x{k}" if k != "-" else ""
        a1.text(5, y, f"{lab}: {op} {ktxt}  →  {ch} canais  ×{rep}   [{res}]", ha="center", va="center", fontsize=9.5)
        if lab != "Classificador (nosso)":
            a1.annotate("", xy=(5, y - 0.5), xytext=(5, y - 0.42), arrowprops=dict(arrowstyle="->"))
        y -= 1.12
    a2.set_title("Bloco MBConv6 (k×k) com Squeeze-and-Excitation", fontsize=12, weight="bold")
    steps = [("Entrada  (H x W x C)", "#e2f0d9"), ("Conv 1x1 expansão (C → 6C) + BN + Swish", "#9dc3e6"),
             ("Conv depthwise k×k + BN + Swish", "#9dc3e6"),
             ("Squeeze-and-Excitation:\nGAP → Dense (redução) → Dense → sigmoid → × canais", "#ffe699"),
             ("Conv 1x1 projeção (6C → C') + BN  (linear)", "#9dc3e6"),
             ("+ conexão residual (se stride 1 e C = C')\ncom drop connect no treino", "#f8cbad"),
             ("Saída  (H' x W' x C')", "#e2f0d9")]
    y = 11
    for txt, col in steps:
        a2.add_patch(FancyBboxPatch((1, y - 0.55), 8, 1.05, boxstyle="round,pad=0.05", fc=col, ec="black"))
        a2.text(5, y, txt, ha="center", va="center", fontsize=9.5)
        if txt.startswith("Saída") is False:
            a2.annotate("", xy=(5, y - 0.95), xytext=(5, y - 0.6), arrowprops=dict(arrowstyle="->"))
        y -= 1.55
    a2.annotate("", xy=(9.3, 2.1), xytext=(9.3, 11), arrowprops=dict(arrowstyle="->", ls="--", color="gray"))
    a2.text(9.45, 6.5, "atalho", rotation=90, color="gray", va="center")
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "arquitetura_efficientnetb1.png", dpi=130); plt.show()

draw_architecture()

In [ ]:
# Conferência no modelo real: blocos por estágio, formato de saída e parâmetros
_b1 = keras.applications.EfficientNetB1(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
_b0 = keras.applications.EfficientNetB0(include_top=False, weights=None, input_shape=(224, 224, 3))

def stage_table(model):
    rows = []
    for s in range(1, 8):
        blocks = sorted({m.group(1) for l in model.layers if (m := re.match(rf"^block{s}([a-z])_", l.name))})
        last = model.get_layer(f"block{s}{blocks[-1]}_project_bn")
        params = sum(l.count_params() for l in model.layers if l.name.startswith(f"block{s}"))
        rows.append({"estágio": s, "blocos": len(blocks), "saída": tuple(last.output.shape[1:]), "parâmetros": params})
    return pd.DataFrame(rows).set_index("estágio")

t1, t0 = stage_table(_b1), stage_table(_b0)
cmp = t1.join(t0[["blocos", "parâmetros"]], rsuffix="_B0").rename(columns={"blocos": "blocos_B1", "parâmetros": "parâmetros_B1"})
display(cmp)
print(f"Blocos MBConv: B0 = {t0.blocos.sum()}, B1 = {t1.blocos.sum()}")
print(f"Parâmetros sem o topo ImageNet: B0 = {_b0.count_params():,}, B1 = {_b1.count_params():,}")
print("Primeiras camadas da B1 (normalização embutida):", [l.name for l in _b1.layers[1:4]])
print("Última camada convolucional (usada no Grad-CAM):", _b1.layers[-1].name, tuple(_b1.layers[-1].output.shape[1:]))
del _b1, _b0; gc.collect()

**📝 Interpretação**

A conferência no modelo Keras **bate com o diagrama e com o artigo**:
- A B1 tem **23 blocos MBConv** (2, 3, 3, 4, 4, 5 e 2 por estágio), contra **16 da B0**.
- Sem o classificador do ImageNet, a B1 tem **6.575.239 parâmetros**, contra 4.049.571 da B0 (+62%).
- Os canais de cada estágio são os mesmos nas duas versões (coeficiente de largura 1,0). O crescimento vem da **profundidade** (coeficiente 1,1) e da **resolução** (240 px em vez de 224).
- A última camada, `top_activation`, gera um mapa **8×8×1280**. É nela que aplicamos o Grad-CAM, então cada célula do mapa corresponde a uma região de ~30×30 px da imagem de entrada: os mapas mostram **regiões**, não detalhes finos.

**Os três conceitos centrais:**
- **Escalonamento composto:** em vez de aumentar só uma dimensão (como ir da ResNet-50 para a ResNet-152, só com mais camadas), a EfficientNet aumenta profundidade, largura e resolução **juntas**, na proporção fixada por α = 1,2, β = 1,1 e γ = 1,15. A lógica: uma imagem maior tem mais detalhes, então precisa de mais camadas (para "enxergar" regiões maiores) e de mais canais (para representar mais padrões).
- **Por que a convolução *depthwise* é barata:** uma convolução comum 3×3 que leva 144 canais a 144 canais faz 3·3·144·144 ≈ 186 mil multiplicações por pixel. A *depthwise* aplica um filtro 3×3 por canal (3·3·144 ≈ 1,3 mil), e a convolução 1×1 seguinte mistura os canais (144·144 ≈ 20,7 mil). O total é ≈ 22 mil, cerca de **8 vezes menos**, com desempenho parecido.
- **O que o *Squeeze-and-Excitation* faz:** resume cada canal em um número (a média global), passa esse vetor por duas camadas densas que aprendem **quais canais importam para aquela imagem** e multiplica cada canal por um peso entre 0 e 1. É uma "atenção" por canal. No nosso problema, por exemplo, canais que respondem a manchas alaranjadas *podem* ser reforçados em imagens de ferrugem.

---
## 3. Dataset: BRACOL (versão *symptom*)

### 3.1 Contexto, o que se classifica e finalidade

- **Contexto:** o Brasil é o maior produtor de café do mundo. Doenças e pragas nas folhas reduzem a fotossíntese e a produtividade, e o diagnóstico em campo depende de especialistas.
- **O que se classifica:** recortes de folhas de café arábica contendo **um sintoma**, em 5 classes: **saudável**, **bicho-mineiro** (praga: larva de mariposa), **ferrugem** (fungo *Hemileia vastatrix*), **mancha de phoma** e **cercosporiose** (fungos).
- **Finalidade possível:** um aplicativo de apoio ao diagnóstico para produtores e técnicos, que identifica o sintoma por uma foto e sugere o encaminhamento.

### 3.2 Fonte e justificativa

- Dataset: KROHLING, R. A.; ESGARIO, J. G. M.; VENTURA, J. A. *BRACOL — A Brazilian Arabica Coffee Leaf images dataset to identification and quantification of coffee diseases and pests*. Mendeley Data, v1, 2019. DOI: [10.17632/yy2k5y8mxg.1](https://doi.org/10.17632/yy2k5y8mxg.1).
- Artigo: ESGARIO, J. G. M.; KROHLING, R. A.; VENTURA, J. A. *Deep learning for classification and severity estimation of coffee leaf biotic stress*. Computers and Electronics in Agriculture, v. 169, 105162, 2020.
- **Por que o GitHub dos autores e não o arquivo do DOI:** o zip do Mendeley confere com o sha256 publicado, mas está **truncado** (não abre, contém só parte das folhas inteiras e nenhuma imagem *symptom*). A auditoria abaixo mostra isso. A versão *symptom* usada no artigo está no repositório dos mesmos autores: [github.com/esgario/lara2018](https://github.com/esgario/lara2018). Segundo o README deles, ela combina imagens próprias com imagens do repositório **Digipathos (Embrapa)**.
- **Por que este dataset:** cumpre a lauda (2.722 imagens, menor classe com 256), é pouco documentado em comparação com outros candidatos e tem contexto brasileiro com erros de gravidade diferente. Também mostrou espaço para análise no piloto: F1 macro de ~0,87 com a base congelada da MobileNetV2, com confusões interpretáveis.

In [ ]:
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

if AUDIT_DOI:
    try:
        api = "https://data.mendeley.com/public-api/datasets/yy2k5y8mxg/files?folder_id=root&version=1"
        f0 = requests.get(api, timeout=60).json()[0]
        cd = f0["content_details"]
        p = DATA_DIR / "bracol_doi.zip"
        if not p.exists():
            with requests.get(cd["download_url"], stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(p, "wb") as fh:
                    for part in r.iter_content(1 << 20):
                        fh.write(part)
        data = p.read_bytes()
        names, i = [], 0
        while (j := data.find(b"PK\x03\x04", i)) >= 0:
            n = struct.unpack("<H", data[j + 26:j + 28])[0]
            names.append(data[j + 30:j + 30 + n].decode("utf-8", "replace")); i = j + 30 + n
        display(pd.Series({
            "arquivo no DOI": f0["filename"], "tamanho (MB)": round(cd["size"] / 1e6, 1),
            "sha256 confere com o publicado": sha256_of(p) == cd["sha256_hash"],
            "zip abre normalmente": zipfile.is_zipfile(p),
            "tem diretório central": data.rfind(b"PK\x05\x06") >= 0,
            "imagens encontradas": sum(n.lower().endswith(".jpg") for n in names),
            "contém imagens 'symptom'": any("symptom" in n.lower() for n in names),
        }, name="auditoria do arquivo do DOI").to_frame())
        p.unlink()
    except Exception as e:
        print("Auditoria não pôde ser feita:", repr(e))

### 3.3 Download e leitura

Baixamos só a pasta `symptom` do repositório (*sparse checkout*, ~41 MB). A divisão train/val/test dos autores é **ignorada**: guardamos a pasta de origem apenas para medir o vazamento que ela teria (Seção 3.5).

A partir do nome do arquivo, extraímos duas informações:
- `{folha}_{x}_{y}_{w}_{h}.jpg` = recorte das imagens dos autores. O primeiro número identifica a **folha**, e vários recortes podem vir da mesma folha.
- `ferr*`, `cerc*`, `bmin*`, `phom*` = imagens do **Digipathos (Embrapa)**.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png"}
LABELS = {"1_health": "saudavel", "2_miner": "bicho_mineiro", "3_rust": "ferrugem",
          "4_phoma": "mancha_phoma", "5_cercospora": "cercosporiose"}

def list_images(root):
    root = Path(root)
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS
                  and "__MACOSX" not in p.parts and not p.name.startswith(".")) if root.exists() else []

repo = DATA_DIR / "lara2018"
SYM = Path(BRACOL_MANUAL_DIR) if BRACOL_MANUAL_DIR else repo / "classification" / "dataset" / "symptom"
if not BRACOL_MANUAL_DIR and len(list_images(SYM)) < 2700:
    if repo.exists():
        shutil.rmtree(repo)
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/esgario/lara2018.git", str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "sparse-checkout", "set", "classification/dataset/symptom"], check=True)

pat = re.compile(r"^(\d+)_(\d+)_(\d+)_(\d+)_(\d+)$")
rows = []
for p in list_images(SYM):
    m = pat.match(p.stem)
    parent2 = p.parent.parent.name
    rows.append({"path": str(p), "label": LABELS.get(p.parent.name, p.parent.name),
                 "orig_split": parent2 if parent2 in {"train", "val", "test"} else "—",
                 "source": "BRACOL (autores)" if m else "Digipathos (Embrapa)",
                 "group": f"folha_{m.group(1)}" if m else f"img_{p.stem}"})
assert len(rows) > 0, f"Nenhuma imagem encontrada em {SYM} — confira o download ou BRACOL_MANUAL_DIR"
DF = pd.DataFrame(rows)
CLASSES = sorted(DF.label.unique())
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
K = len(CLASSES)
print(f"{len(DF)} imagens, {K} classes: {CLASSES}")

### 3.4 Análise inicial

Uma única leitura de cada imagem gera: formato, modo de cor, canais, resolução, md5 (duplicatas exatas), pHash (quase-duplicatas) e uma miniatura 48×48 (confirmação das duplicatas e imagem média por classe).

In [ ]:
THUMB = 48

def scan_one(path):
    with open(path, "rb") as fh:
        raw = fh.read()
    with Image.open(io.BytesIO(raw)) as im:
        info = {"path": path, "md5": hashlib.md5(raw).hexdigest(), "format": im.format, "mode": im.mode,
                "channels": len(im.getbands()), "width": im.size[0], "height": im.size[1]}
        rgb = im.convert("RGB")
    small = rgb.resize((128, 128), Image.Resampling.BILINEAR)
    info["brightness"] = float(np.asarray(small, dtype=np.float32).mean())
    thumb = np.asarray(small.resize((THUMB, THUMB), Image.Resampling.BILINEAR), dtype=np.uint8)
    bits = np.asarray(imagehash.phash(small).hash, dtype=np.uint8).ravel()
    return info, thumb, bits

with ThreadPoolExecutor(8) as ex:
    res = list(ex.map(scan_one, DF.path.tolist()))
META = pd.DataFrame([r[0] for r in res])
THUMBS = np.stack([r[1] for r in res])
BITS = np.stack([r[2] for r in res])
DF = DF.merge(META, on="path", how="left")
DF["aspect"] = (DF.width / DF.height).round(2)

cnt = DF.label.value_counts()
display(pd.DataFrame({
    "imagens": len(DF), "classes": K,
    "maior classe": f"{cnt.idxmax()} ({cnt.max()})", "menor classe": f"{cnt.idxmin()} ({cnt.min()})",
    "razão maior/menor": round(cnt.max() / cnt.min(), 2),
    "cumpre ≥1.000 imagens": len(DF) >= 1000, "cumpre ≥100 por classe": bool(cnt.min() >= 100),
    "formatos": str(dict(Counter(DF.format))), "modos de cor": str(dict(Counter(DF["mode"]))),
    "canais": str(sorted(DF.channels.unique().tolist())),
    "largura (mín–máx)": f"{DF.width.min()}–{DF.width.max()}", "altura (mín–máx)": f"{DF.height.min()}–{DF.height.max()}",
}, index=["BRACOL symptom"]).T)

fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
ax[0].barh(cnt.index[::-1], cnt.values[::-1], color="#4C72B0"); ax[0].axvline(100, c="red", ls="--", lw=1)
ax[0].set_title("Imagens por classe (linha: mínimo da lauda)")
for c in CLASSES:
    ax[1].hist(DF.loc[DF.label == c, "width"], bins=30, alpha=0.5, label=c)
ax[1].set_title("Largura (px) por classe"); ax[1].legend(fontsize=7)
DF.boxplot(column="aspect", by="label", ax=ax[2], rot=30); ax[2].set_title("Proporção largura/altura"); fig.suptitle("")
plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "analise_inicial.png", dpi=110); plt.show()

display(DF.groupby("label").agg(n=("path", "size"), largura_mediana=("width", "median"),
                                altura_mediana=("height", "median"), brilho_medio=("brightness", "mean")).round(1))

In [ ]:
# Grade com 5 amostras por classe e imagem média por classe
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(K, 6, figsize=(14, 2.4 * K))
for r, c in enumerate(CLASSES):
    sub = DF[DF.label == c]
    for k, i in enumerate(rng.choice(len(sub), 5, replace=False)):
        row = sub.iloc[i]
        axes[r, k].imshow(Image.open(row.path).convert("RGB"))
        axes[r, k].set_title(f"{row.width}x{row.height} · {row.source.split()[0]}", fontsize=7)
    axes[r, 5].imshow(THUMBS[DF.label.values == c].mean(0).astype(np.uint8)); axes[r, 5].set_title("média da classe", fontsize=7)
    axes[r, 0].set_ylabel(c, rotation=0, ha="right", va="center", fontsize=9)
    for a in axes[r]: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "grade_amostras.png", dpi=110); plt.show()

### 3.5 Qualidade: fontes misturadas, recortes da mesma folha e duplicatas

**Por que isso importa:** se recortes da mesma folha (mesma iluminação e textura) caírem em treino e em teste, o teste fica "fácil" demais e o resultado sai otimista (**vazamento**). E se uma fonte de imagens aparecer só em algumas classes, o modelo pode aprender o **estilo da foto** em vez da lesão (**atalho**).

In [ ]:
display(Markdown("**Fonte das imagens por classe:**"))
display(pd.crosstab(DF.source, DF.label, margins=True))

leaf = DF[DF.group.str.startswith("folha_")]
per_leaf = leaf.group.value_counts()
display(pd.Series({
    "folhas distintas (imagens dos autores)": per_leaf.size,
    "folhas com mais de 1 recorte": int((per_leaf > 1).sum()),
    "máximo de recortes por folha": int(per_leaf.max()),
    "folhas com recortes de mais de uma classe": int((leaf.groupby("group").label.nunique() > 1).sum()),
}, name="recortes por folha").to_frame())

def find_pairs(bits, thr, chunk=2048):
    A = bits.astype(np.float32); s = A.sum(1); out = []
    for i0 in range(0, len(A), chunk):
        dist = s[i0:i0 + chunk, None] + s[None, :] - 2 * (A[i0:i0 + chunk] @ A.T)
        ii, jj = np.nonzero(dist <= thr + 0.5); ii = ii + i0
        keep = jj > ii
        out.append(np.stack([ii[keep], jj[keep]], 1))
    return np.concatenate(out).astype(int)

def ncc(G, pairs):
    if len(pairs) == 0:
        return np.zeros(0)
    a = G[pairs[:, 0]].reshape(len(pairs), -1); b = G[pairs[:, 1]].reshape(len(pairs), -1)
    a = a - a.mean(1, keepdims=True); b = b - b.mean(1, keepdims=True)
    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-6)

G = THUMBS.astype(np.float32).mean(-1)
cand = find_pairs(BITS, HASH_THR)
NEAR = cand[ncc(G, cand) >= NCC_THR] if len(cand) else cand
y_all = DF.label.values
display(pd.Series({
    "imagens em duplicatas exatas (md5)": int(DF.duplicated("md5", keep=False).sum()),
    "pares quase-duplicados confirmados": len(NEAR),
    "... com rótulos diferentes": int((y_all[NEAR[:, 0]] != y_all[NEAR[:, 1]]).sum()) if len(NEAR) else 0,
}, name="duplicatas").to_frame())

if len(NEAR):
    show = NEAR[:6]
    fig, axes = plt.subplots(len(show), 2, figsize=(5, 2.4 * len(show)))
    axes = np.atleast_2d(axes)
    for r, (i, j) in enumerate(show):
        for c, idx in enumerate((i, j)):
            axes[r, c].imshow(Image.open(DF.path[idx]).convert("RGB")); axes[r, c].axis("off")
            axes[r, c].set_title(f"{DF.label[idx]}\n{Path(DF.path[idx]).name}", fontsize=7)
    plt.suptitle("Quase-duplicatas confirmadas (conferir visualmente)", fontsize=9); plt.tight_layout(); plt.show()

### 3.6 Divisão treino / validação / teste (holdout estratificado por grupos)

**Decisões:**
- **70 / 15 / 15**, estratificada por classe e **por grupo**: todos os recortes da mesma folha, e imagens ligadas por quase-duplicata, ficam no **mesmo** conjunto. Implementação: `StratifiedGroupKFold` com 20 folds e seed fixa (14 folds = treino, 3 = validação, 3 = teste).
- **A divisão oficial dos autores não é usada**, porque espalha recortes da mesma folha entre os conjuntos. A tabela abaixo quantifica isso.
- **O conjunto de teste fica reservado desde já:** a divisão é salva em CSV, e a função que monta os dados **recusa** o split `test` enquanto `TEST_UNLOCKED = False`. O teste só será usado na etapa 7 da lauda, depois de escolhido o modelo recomendado.

In [ ]:
parent = np.arange(len(DF))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
for i, j in NEAR:
    union(int(i), int(j))
for idx in DF.groupby("group").indices.values():
    for k in idx[1:]:
        union(int(idx[0]), int(k))
DF["split_group"] = [f"g{find(i)}" for i in range(len(DF))]

sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
fold = np.full(len(DF), -1)
for k, (_, f) in enumerate(sgkf.split(DF.path, DF.label, DF.split_group)):
    fold[f] = k
DF["split"] = np.where(fold < 3, "test", np.where(fold < 6, "val", "train"))

def groups_crossing(split_col):
    s = DF.groupby("group")[split_col].nunique()
    return int((s > 1).sum())
check = {"divisão oficial dos autores": groups_crossing("orig_split") if (DF.orig_split != "—").all() else None,
         "nossa divisão por grupos": groups_crossing("split")}
display(pd.Series(check, name="folhas com recortes em mais de um conjunto").to_frame())
assert check["nossa divisão por grupos"] == 0

prop = pd.crosstab(DF.label, DF.split)[["train", "val", "test"]]
prop_pct = (prop / prop.sum() * 100).round(1).astype(str) + "%"
display(Markdown("**Imagens por classe e conjunto (contagem e proporção da coluna):**"))
display(prop.astype(str) + "  (" + prop_pct + ")")

SPLIT_CSV = OUT_DIR / f"divisao_bracol_seed{SEED}.csv"
DF[["path", "label", "source", "group", "split_group", "split"]].to_csv(SPLIT_CSV, index=False)
print("Divisão salva em", SPLIT_CSV)

TEST_UNLOCKED = False
def get_split(name):
    if name == "test" and not TEST_UNLOCKED:
        raise PermissionError("Conjunto de teste reservado: só é liberado na etapa 7, depois da escolha do modelo.")
    return DF[DF.split == name].reset_index(drop=True)

TRAIN, VAL = get_split("train"), get_split("val")
print(f"treino = {len(TRAIN)}, validação = {len(VAL)}, teste reservado = {(DF.split == 'test').sum()} (não carregado)")

**📝 Interpretação da análise dos dados**

1. **Conformidade com a lauda.** São 2.722 imagens, todas JPEG RGB de 3 canais, em 5 classes. A maior é ferrugem (991) e a menor é saudável (256), uma razão de **3,87**. O dataset cumpre os mínimos da lauda (≥ 1.000 imagens e ≥ 100 por classe), mas o desbalanceamento justifica usar **F1 macro** como métrica principal e **pesos de classe** na perda.

2. **Fontes misturadas: principal risco de atalho.**
   - 592 imagens (21,7%) vêm do Digipathos/Embrapa, e **nenhuma é da classe saudável**. Na ferrugem, elas são 383 de 991 (39%).
   - As imagens do Digipathos são **menores** (largura mediana de 115 px, contra 222 px nas dos autores) e **mais claras** (brilho médio 107,6 × 93,3).
   - Mais grave: **todas as 150 imagens com menos de 64 px são do Digipathos, e 132 delas são ferrugem.** Ampliadas para 240×240, essas imagens ficam borradas.
   - Por isso, o modelo pode aprender **"imagem pequena e borrada ⇒ ferrugem"** e **"foto dos autores ⇒ pode ser saudável"**, em vez de aprender a lesão.
   - **Hipótese a testar:** a taxa de erro por fonte (Seção 6) e o Grad-CAM vão mostrar se o modelo depende da fonte.

3. **Diferenças de brilho e tamanho por classe.** O bicho-mineiro tem as imagens mais escuras (brilho 80,7), e a mancha de phoma tem os maiores recortes (mediana de 231 px). Isso pode refletir a própria lesão ou as condições de captura. Por isso a augmentation de contraste, que reduz a dependência de brilho.

4. **Vários recortes da mesma folha.** As imagens dos autores vêm de **1.344 folhas**. Dessas, **514 têm mais de um recorte** (até 8), e **101 têm recortes de classes diferentes** (folha com mais de um sintoma). A divisão oficial dos autores deixa **286 folhas em mais de um conjunto**: um modelo poderia "reconhecer a folha" no teste, e isso inflaria o resultado. Na nossa divisão por grupos, são **0**. Esperamos números um pouco menores que os do artigo, mas honestos.

5. **Duplicatas: poucas, mas uma merece atenção.**
   - **Duplicata exata em cercosporiose:** `1631_1449_0489_0327_0347.jpg` e `1632_1449_0489_0327_0347.jpg` têm as mesmas coordenadas e números de folha consecutivos, ou seja, a mesma foto registrada duas vezes. Foram agrupadas e ficaram juntas no mesmo conjunto.
   - **Quase-duplicata com rótulos diferentes:** `141_1345_0382_0189_0168.jpg` (ferrugem) e `321_1227_0207_0235_0227.jpg` (saudável) são de folhas diferentes, mas visualmente quase idênticas. Pode ser **ruído de rótulo**, ou um recorte de ferrugem com lesão quase imperceptível. O impacto é pequeno (2 de 2.722 imagens), mas é um exemplo concreto de ambiguidade nos rótulos (ver a figura de pares acima).

6. **Divisão estratificada por grupos.** Treino, validação e teste ficaram com **1.916 / 388 / 418** imagens (70,4% / 14,3% / 15,4%). Como cada folha precisa ficar inteira num só conjunto, a estratificação é aproximada: a ferrugem tem 36,8% / 34,8% / 36,1%, e a classe saudável 9,0% / 9,0% / 11,5%. Com só **35 imagens saudáveis na validação**, cada erro nessa classe muda o recall em ~2,9 pontos, e os intervalos de confiança refletem isso.
   - **Reprodutibilidade:** a divisão depende da versão do scikit-learn (no Colab, 1.6.1). Por isso o arquivo `divisao_bracol_seed42.csv` salvo no Drive é a referência oficial e deve ser reutilizado.

---
## 4. Pré-processamento e métricas

### 4.1 Pré-processamento esperado pelos pesos

| Decisão | Justificativa |
|---|---|
| **Redimensionar para 240×240** (bilinear com *antialias*) | 240 é a resolução com que a B1 foi treinada no ImageNet. Os recortes variam de 15 a 852 px; os pequenos são ampliados (e ficam borrados) e os grandes são reduzidos |
| **Redimensionamento direto**, sem preservar a proporção | Preencher a imagem com bordas pretas criaria um padrão artificial ligado à proporção do recorte, que difere entre as fontes. Uma deformação leve preserva cor e textura da lesão, que é o que importa |
| **3 canais RGB** | Todas as imagens são JPEG RGB (ver 3.4). `decode_image(channels=3)` garante isso mesmo se surgir alguma em tons de cinza |
| **Sem normalização manual** | A EfficientNet do Keras já tem *Rescaling* + *Normalization* com as estatísticas do ImageNet **dentro do modelo**, e espera pixels de 0 a 255. Por isso `efficientnet.preprocess_input` não faz nada. Normalizar de novo deixaria a entrada fora da distribuição dos pesos pré-treinados |

### 4.2 *Data augmentation* (só no treino)

Viradas horizontal e vertical, rotação de até ±90°, zoom de ±10% e contraste de ±10%. Uma folha fotografada de cima não tem orientação "certa", então essas transformações **não mudam a classe** e reduzem o overfitting num treino de ~1.900 imagens. A validação e o teste **nunca** recebem augmentation.

### 4.3 Métricas, erros mais graves e regra de seleção

**Erros mais prejudiciais neste contexto:**
1. **Doença ou praga classificada como "saudável" (falso saudável):** nada é feito, e a ferrugem e o bicho-mineiro se espalham. É o erro **mais grave**.
2. **Praga × doença** (bicho-mineiro ↔ fungos): leva ao produto errado, inseticida no lugar de fungicida ou o contrário.
3. **Doença × doença** (ex.: ferrugem ↔ cercosporiose): o manejo é diferente, mas ao menos a planta recebe atenção.
4. **Saudável classificada como doente:** gera custo com aplicação desnecessária, mas é o erro menos grave.

**Métricas escolhidas:**
- **F1 macro** (média do F1 das 5 classes, todas com o mesmo peso): é a **métrica principal e a regra de seleção**. A acurácia sozinha seria dominada pela ferrugem (36% das imagens).
- ***Balanced accuracy*** (média dos *recalls*): confirma o desempenho por classe, sem efeito do desbalanceamento.
- ***Recall* por classe**, com atenção às classes doentes.
- **Taxa de falso saudável**: fração das imagens doentes previstas como saudáveis. É a métrica do erro mais grave.
- **Intervalo de confiança de 95% por bootstrap**: a validação tem ~400 imagens, então diferenças pequenas podem ser ruído.
- **Matriz de confusão normalizada**: mostra **quais** classes se confundem.

**Regra de seleção** (a mesma na base congelada e nas estratégias A e B): escolher a época com **maior F1 macro na validação**, com *early stopping* (paciência de 4 épocas) e restauração dos melhores pesos.

**Pesos de classe:** cada classe pesa *n_total / (K · n_classe)* na perda, para o modelo não favorecer a ferrugem. Isso combina com a escolha do F1 macro.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
PREPROCESS = keras.applications.efficientnet.preprocess_input   # identidade: a normalização está no modelo

AUGMENT = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.25, fill_mode="reflect", seed=SEED),
    layers.RandomZoom(0.1, fill_mode="reflect", seed=SEED),
    layers.RandomContrast(0.1, seed=SEED),
], name="augmentation")

def load_img(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear", antialias=True)
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)

def make_ds(frame, training):
    y = keras.utils.to_categorical(frame.label.map(CLASS_TO_ID).values, K).astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, y))
    ds = ds.map(lambda p, t: (load_img(p), t), num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH).map(lambda x, t: (PREPROCESS(tf.cast(x, tf.float32)), t), num_parallel_calls=AUTOTUNE)
    if training and USE_AUGMENTATION:
        ds = ds.map(lambda x, t: (AUGMENT(x, training=True), t), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

TRAIN_DS, VAL_DS = make_ds(TRAIN, True), make_ds(VAL, False)
Y_VAL = VAL.label.map(CLASS_TO_ID).values

counts = TRAIN.label.value_counts().reindex(CLASSES)
CLASS_WEIGHT = ({i: len(TRAIN) / (K * counts[c]) for i, c in enumerate(CLASSES)} if USE_CLASS_WEIGHTS else None)
display(pd.DataFrame({"imagens no treino": counts.values,
                      "peso na perda": [round(CLASS_WEIGHT[i], 3) if CLASS_WEIGHT else 1 for i in range(K)]}, index=CLASSES))

# Exemplos do que o modelo vê no treino (mesma imagem com augmentations diferentes)
ex = load_img(TRAIN.path[0])[None]
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(ex[0].numpy()); axes[0].set_title(f"original 240x240\n{TRAIN.label[0]}", fontsize=8)
for a in axes[1:]:
    a.imshow(np.clip(AUGMENT(tf.cast(ex, tf.float32), training=True)[0].numpy(), 0, 255).astype(np.uint8))
    a.set_title("augmentation", fontsize=8)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

---
## 5. Transfer learning com a base congelada (condição de referência)

**Arquitetura adaptada:** EfficientNetB1 sem o topo ImageNet (`include_top=False`) → *GlobalAveragePooling2D* → *Dropout* 0,3 → *Dense* com 5 saídas e softmax.

**Decisões:**
- **Base inteira congelada** (`trainable=False`); só o classificador novo aprende.
- A base é chamada com `training=False`, então as camadas de *BatchNormalization* usam sempre as estatísticas do ImageNet. Isso será mantido no fine-tuning: é a recomendação oficial do Keras para EfficientNet, porque atualizar a BN com *batches* de 32 imagens destruiria o que foi pré-treinado.
- **Hiperparâmetros:** Adam, lr = 1e-3, batch 32, perda *categorical cross-entropy* com pesos de classe, até 20 épocas com *early stopping* (paciência de 4) pelo **F1 macro de validação**.
- Os **melhores pesos são salvos** e serão o ponto de partida **idêntico** das estratégias A e B.

In [ ]:
def build_model():
    base = keras.applications.EfficientNetB1(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    head = keras.Sequential([layers.GlobalAveragePooling2D(name="gap"),
                             layers.Dropout(DROPOUT, seed=SEED, name="dropout"),
                             layers.Dense(K, activation="softmax", name="pred")], name="classificador")
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="imagem_rgb_0_255")
    out = head(base(inp, training=False))
    return keras.Model(inp, out, name="efficientnetb1_bracol"), base, head

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="categorical_crossentropy",
                  metrics=["accuracy", keras.metrics.F1Score(average="macro", name="f1_macro")])

def n_trainable(model):
    return int(sum(np.prod(v.shape) for v in model.trainable_weights))

class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.perf_counter() - self.t0)

def fit(model, epochs, weights_path, initial_epoch=0, best_so_far=None):
    timer = EpochTimer()
    cbs = [timer,
           keras.callbacks.EarlyStopping(monitor="val_f1_macro", mode="max", patience=PATIENCE, restore_best_weights=True),
           keras.callbacks.ModelCheckpoint(str(weights_path), monitor="val_f1_macro", mode="max",
                                           save_best_only=True, save_weights_only=True,
                                           initial_value_threshold=best_so_far)]
    h = model.fit(TRAIN_DS, validation_data=VAL_DS, epochs=initial_epoch + epochs, initial_epoch=initial_epoch,
                  class_weight=CLASS_WEIGHT, callbacks=cbs, verbose=2)
    return h, timer.times

keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
MODEL, BASE, HEAD = build_model()
compile_model(MODEL, LR_FROZEN)
ARCH = {"entrada": MODEL.input_shape, "saída": MODEL.output_shape, "parâmetros totais": MODEL.count_params(),
        "parâmetros treináveis": n_trainable(MODEL),
        "parâmetros congelados": MODEL.count_params() - n_trainable(MODEL), "camadas da base": len(BASE.layers),
        "otimizador": "Adam", "learning rate": LR_FROZEN, "batch size": BATCH, "épocas (máx.)": MAX_EPOCHS_FROZEN,
        "early stopping": f"val_f1_macro, paciência {PATIENCE}", "dropout": DROPOUT,
        "pesos de classe": USE_CLASS_WEIGHTS, "augmentation": USE_AUGMENTATION}
display(pd.Series({k: (f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)) for k, v in ARCH.items()},
                  name="modelo adaptado — base congelada").to_frame())
HEAD.summary()

In [ ]:
W_FROZEN = OUT_DIR / "modelos" / "base_congelada.weights.h5"
H_FROZEN, T_FROZEN = fit(MODEL, MAX_EPOCHS_FROZEN, W_FROZEN)
MODEL.load_weights(W_FROZEN)          # garante os pesos da melhor época
MODEL.save(OUT_DIR / "modelos" / "base_congelada.keras")
with open(OUT_DIR / "modelos" / "classes_e_preprocessamento.json", "w") as f:
    json.dump({"classes": CLASSES, "indice_para_classe": {i: c for i, c in enumerate(CLASSES)},
               "img_size": IMG_SIZE, "entrada": "RGB 0-255, redimensionamento bilinear direto; normalização dentro do modelo",
               "arquitetura": "EfficientNetB1 (keras.applications), include_top=False + GAP + Dropout + Dense softmax"},
              f, indent=2, ensure_ascii=False)
best_ep = int(np.argmax(H_FROZEN.history["val_f1_macro"])) + 1
print(f"Melhor época: {best_ep} de {len(T_FROZEN)} | tempo total {sum(T_FROZEN)/60:.1f} min "
      f"| época 1: {T_FROZEN[0]:.0f}s (inclui leitura/cache), demais (mediana): {np.median(T_FROZEN[1:]) if len(T_FROZEN) > 1 else T_FROZEN[0]:.0f}s")
print("Pesos salvos em", W_FROZEN)

In [ ]:
def plot_curves(hist, title, marks=()):
    H = hist if isinstance(hist, dict) else hist.history
    ep = np.arange(1, len(H["loss"]) + 1)
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
    for a, key, lab in [(ax[0], "loss", "loss"), (ax[1], "accuracy", "acurácia"), (ax[2], "f1_macro", "F1 macro")]:
        a.plot(ep, H[key], "o-", label="treino"); a.plot(ep, H["val_" + key], "s-", label="validação")
        for m in marks: a.axvline(m + 0.5, ls="--", c="gray")
        a.set_xlabel("época"); a.set_title(f"{title} — {lab}"); a.legend(fontsize=8)
    best = int(np.argmax(H["val_f1_macro"]))
    ax[2].scatter([best + 1], [H["val_f1_macro"][best]], s=120, facecolors="none", edgecolors="red", label="selecionada")
    ax[2].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / f"curvas_{title.replace(' ', '_')}.png", dpi=110); plt.show()

plot_curves(H_FROZEN, "base congelada")

**📝 Interpretação das curvas**

- O treino usou as 20 épocas do teto, e a **melhor época foi a 17** (F1 macro de validação 0,903). Nas últimas épocas o F1 ainda oscilava perto do máximo (0,879 → 0,897 → 0,899), por isso o early stopping (paciência 4) não chegou a interromper. A base congelada **ainda não tinha saturado totalmente**, mas o ganho por época já era pequeno.
- **Sem sinal de overfitting:** a loss de validação caiu de forma contínua (0,708 → 0,244) e terminou no mesmo nível da de treino (0,213). O F1 de treino (0,92) e o de validação (0,90) ficaram próximos.
- Nas primeiras épocas, a validação ficou **acima** do treino (época 1: F1 0,73 × 0,57). Isso é esperado: o treino tem augmentation e dropout, que deixam a tarefa mais difícil, e a validação não.
- **Custo:** apenas 6.405 parâmetros treináveis (a camada Dense: 1.280 × 5 + 5). Cada época levou ~27 s na T4; a primeira levou 188 s porque inclui a leitura e o cache das imagens. Total de 13,4 min.

In [ ]:
def evaluate(model, ds, y_true, tag, split_name="VALIDAÇÃO"):
    prob = model.predict(ds, verbose=0); pred = prob.argmax(1)
    rng = np.random.default_rng(SEED); n = len(y_true); bf, bb = [], []
    for _ in range(N_BOOT):
        i = rng.integers(0, n, n)
        bf.append(f1_score(y_true[i], pred[i], average="macro", labels=range(K), zero_division=0))
        bb.append(balanced_accuracy_score(y_true[i], pred[i]))
    sau = CLASS_TO_ID["saudavel"]; doente = y_true != sau
    m = {"acurácia": accuracy_score(y_true, pred),
         "F1 macro": f1_score(y_true, pred, average="macro", zero_division=0),
         "F1 macro IC95%": np.percentile(bf, [2.5, 97.5]).round(3).tolist(),
         "balanced accuracy": balanced_accuracy_score(y_true, pred),
         "balanced acc IC95%": np.percentile(bb, [2.5, 97.5]).round(3).tolist(),
         "taxa de falso saudável": float((pred[doente] == sau).mean()),
         "saudável prevista como doente": float((pred[~doente] != sau).mean())}
    display(Markdown(f"**{tag} — métricas ({split_name})**"))
    display(pd.Series({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()}, name=tag).to_frame())
    rep = pd.DataFrame(classification_report(y_true, pred, labels=range(K), target_names=CLASSES,
                                             output_dict=True, zero_division=0)).T.iloc[:K].round(3)
    display(rep)
    cm = confusion_matrix(y_true, pred, labels=range(K), normalize="true")
    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=9, color="white" if cm[i, j] > 0.5 else "black")
    ax.set_xticks(range(K)); ax.set_xticklabels(CLASSES, rotation=40, ha="right"); ax.set_yticks(range(K)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("previsto"); ax.set_ylabel("real"); ax.set_title(f"{tag} — matriz de confusão normalizada ({split_name.lower()})")
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / f"matriz_{tag.replace(' ', '_')}.png", dpi=110); plt.show()
    return m, prob

M_FROZEN, PROB_FROZEN = evaluate(MODEL, VAL_DS, Y_VAL, "base congelada")
RESULTS = {"base congelada": {**M_FROZEN, "parâmetros treináveis": n_trainable(MODEL), "épocas": len(T_FROZEN),
                              "melhor época": best_ep, "tempo (min)": sum(T_FROZEN) / 60}}
with open(OUT_DIR / "resultados_validacao.json", "w") as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False, default=float)

**📝 Interpretação das métricas (validação, 388 imagens)**

- **F1 macro = 0,903** (IC95% 0,870–0,934), **balanced accuracy = 0,914** (0,884–0,945) e acurácia = 0,920, treinando apenas o classificador novo. Os pesos do ImageNet já separam bem os sintomas.
- **Pior classe: cercosporiose** (F1 0,788; precisão 0,765; recall 0,812). É a classe que mais "atrai" erros: recebe 12 imagens de outras classes (5 de bicho-mineiro, 4 de phoma e 3 de ferrugem) e espalha seus próprios 9 erros entre bicho-mineiro, ferrugem e phoma. Suas lesões são manchas marrons com halo, visualmente parecidas com as das outras doenças.
- **Saudável:** recall alto (0,971: só 1 das 35 foi confundida), mas precisão menor (0,829), porque 7 folhas doentes foram classificadas como saudáveis.
- **Erro mais grave (falso saudável): 1,98%** das folhas doentes (7 de 353): 5 de ferrugem e 2 de phoma. É baixo, mas **ferrugem → saudável** está entre as três confusões mais frequentes e é um alvo do fine-tuning. Nossa hipótese é que sejam lesões iniciais, pequenas.
- As demais classes estão bem resolvidas: ferrugem F1 0,951, bicho-mineiro 0,947 e phoma 0,933.
- **Comparação com o piloto:** a MobileNetV2 congelada, com 5 épocas e sem augmentation, chegou a F1 0,870 (em outra divisão). A EfficientNetB1, com mais épocas, augmentation e pesos de classe, chegou a 0,903, e ainda há espaço para o fine-tuning.

---
## 6. Análise visual preliminar (validação, base congelada)

Para cada imagem mostramos a **imagem, a classe real, a classe prevista e o score** (probabilidade da classe prevista), como pede a lauda:
1. os **erros mais confiantes** (o modelo errou com certeza alta, os mais informativos);
2. exemplos das **principais confusões**;
3. um **acerto por classe** e os **acertos mais incertos**;
4. **Grad-CAM** preliminar, para ver *onde* o modelo olha;
5. **erros por fonte** (autores × Digipathos), para testar a hipótese de atalho.

Estas mesmas imagens serão acompanhadas depois do fine-tuning para identificar **erros corrigidos, persistentes e novos**. Os índices ficam salvos em `imagens_acompanhadas.json`.

In [ ]:
PRED_F = PROB_FROZEN.argmax(1); CONF_F = PROB_FROZEN.max(1)

def show(idx, title, ncols=6):
    idx = list(idx)
    if not idx:
        print(f"{title}: nenhum exemplo."); return
    nrows = int(np.ceil(len(idx) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.9 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for a in axes: a.axis("off")
    for a, i in zip(axes, idx):
        ok = PRED_F[i] == Y_VAL[i]
        a.imshow(Image.open(VAL.path[i]).convert("RGB"))
        a.set_title(f"real: {CLASSES[Y_VAL[i]]}\nprev: {CLASSES[PRED_F[i]]} ({CONF_F[i]:.2f})", fontsize=8,
                    color="green" if ok else "red")
    fig.suptitle(title, fontsize=11); plt.tight_layout(); plt.show()

wrong = np.where(PRED_F != Y_VAL)[0]; right = np.where(PRED_F == Y_VAL)[0]
print(f"Erros na validação: {len(wrong)} de {len(Y_VAL)}")
TOP_ERR = wrong[np.argsort(-CONF_F[wrong])][:12]
show(TOP_ERR, "1) Erros mais confiantes (vermelho = erro)")

pairs = pd.Series([(CLASSES[Y_VAL[i]], CLASSES[PRED_F[i]]) for i in wrong]).value_counts().head(3)
display(Markdown("**Principais confusões (real → previsto):**")); display(pairs.rename("erros").to_frame())
CONF_EX = []
for (yt, yp), _ in pairs.items():
    ids = [i for i in wrong if CLASSES[Y_VAL[i]] == yt and CLASSES[PRED_F[i]] == yp][:6]
    CONF_EX += ids
    show(ids, f"2) Confusão {yt} → {yp}")

rng = np.random.default_rng(SEED)
ONE_PER = [int(rng.choice(right[Y_VAL[right] == c])) for c in range(K) if (Y_VAL[right] == c).any()]
show(ONE_PER, "3a) Um acerto por classe (escolhido ao acaso)")
UNCERTAIN = right[np.argsort(CONF_F[right])][:6]
show(UNCERTAIN, "3b) Acertos mais incertos (menor score)")

display(Markdown("**5) Erros por fonte das imagens (validação):**"))
display(pd.DataFrame({"fonte": VAL.source, "erro": PRED_F != Y_VAL}).groupby("fonte").erro
        .agg(n="size", taxa_de_erro="mean").round(3))

In [ ]:
# 4) Grad-CAM preliminar na última camada convolucional da base, sobre o logit da classe prevista
CAM_LAYER = BASE.layers[-1].name

def gradcam(x_uint8, class_idx):
    W, b = [tf.constant(w) for w in HEAD.get_layer("pred").get_weights()]
    x = tf.cast(x_uint8[None], tf.float32)
    with tf.GradientTape() as tape:
        fmap = BASE(PREPROCESS(x), training=False)
        tape.watch(fmap)
        score = (tf.matmul(tf.reduce_mean(fmap, axis=(1, 2)), W) + b)[:, class_idx]
    g = tape.gradient(score, fmap)[0]
    cam = tf.nn.relu(tf.reduce_sum(fmap[0] * tf.reduce_mean(g, axis=(0, 1)), -1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return tf.image.resize(cam[..., None].astype(np.float32), (IMG_SIZE, IMG_SIZE)).numpy()[..., 0]

CAM_IDX = list(TOP_ERR[:3]) + ONE_PER[:3]
fig, axes = plt.subplots(2, len(CAM_IDX), figsize=(2.8 * len(CAM_IDX), 6))
for c, i in enumerate(CAM_IDX):
    img = load_img(VAL.path[i]).numpy()
    cam = gradcam(img, int(PRED_F[i]))
    axes[0, c].imshow(img); axes[0, c].set_title(f"real: {CLASSES[Y_VAL[i]]}", fontsize=8)
    axes[1, c].imshow(np.clip(0.55 * img / 255 + 0.45 * plt.cm.jet(cam)[..., :3], 0, 1))
    axes[1, c].set_title(f"classe investigada: {CLASSES[PRED_F[i]]}\nscore {CONF_F[i]:.2f}", fontsize=8,
                         color="green" if PRED_F[i] == Y_VAL[i] else "red")
    for a in axes[:, c]: a.axis("off")
fig.suptitle(f"Grad-CAM — camada '{CAM_LAYER}' (mapa 8x8), base congelada", fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.96)); plt.savefig(OUT_DIR / "figs" / "gradcam_preliminar.png", dpi=110); plt.show()

TRACKED = sorted(set(map(int, list(TOP_ERR) + CONF_EX + ONE_PER + list(UNCERTAIN))))
with open(OUT_DIR / "imagens_acompanhadas.json", "w") as f:
    json.dump({"indices_validacao": TRACKED, "arquivos": VAL.path[TRACKED].map(lambda p: Path(p).name).tolist()}, f, indent=1)
print(f"{len(TRACKED)} imagens de validação salvas para acompanhar nas condições A e B.")

**📝 Hipóteses preliminares**

1. **Os erros mais confiantes parecem ambiguidades visuais reais, não falhas grosseiras.** O erro mais confiante é uma cercosporiose prevista como mancha de phoma com score 0,99. No Grad-CAM, o mapa cai exatamente sobre a lesão (mancha escura com centro claro), cuja forma lembra a de phoma. Outro exemplo é uma ferrugem prevista como cercosporiose (0,90): a lesão é uma mancha escura com halo amarelo, mais típica de cercosporiose do que das pústulas alaranjadas da ferrugem. Pode ser um estágio avançado da doença ou **ambiguidade no rótulo**.
2. **Há pelo menos um caso de atenção no fundo.** No bicho-mineiro previsto como cercosporiose (0,95), o mapa fica **espalhado pelo fundo branco e pela borda da folha**, e não na mina. Num acerto de bicho-mineiro, além da lesão, aparece um ponto quente no canto da imagem. Recortes com fundo branco são comuns nas imagens dos autores, então vale acompanhar se o modelo usa a borda da folha como pista.
3. **A fonte das imagens influencia o erro.** A taxa de erro é de 7,2% nas imagens dos autores (21 de 292) e de 10,4% nas do Digipathos (10 de 96), e o padrão muda por classe. A ferrugem do Digipathos erra menos (2 de 63, 3%) que a dos autores (6 de 72, 8%), o que é coerente com a hipótese de atalho "imagem pequena e borrada ⇒ ferrugem" (Seção 3). Já a mancha de phoma do Digipathos erra muito (4 de 9). Os números são pequenos, então isso é hipótese, não conclusão.
4. **Os acertos mais incertos** (score entre 0,46 e 0,51) estão em bicho-mineiro, cercosporiose, phoma e ferrugem, justamente as classes que se confundem entre si: o modelo fica "dividido" entre lesões parecidas.
5. **O que esperamos do fine-tuning:** ajustar os blocos mais profundos deve ajudar a separar as **texturas finas** das lesões (cercosporiose × phoma × bicho-mineiro) e a reconhecer **lesões iniciais de ferrugem**, reduzindo o falso saudável. Se, depois do fine-tuning, o Grad-CAM se concentrar mais na lesão e menos no fundo, isso reforça que o ganho veio de características relevantes.

---
## 7. Estratégias de fine-tuning A e B

As duas partem de **cópias dos mesmos pesos salvos** na Seção 5 (`base_congelada.weights.h5`).

| | **Estratégia A: parcial em etapa única** | **Estratégia B: descongelamento progressivo** |
|---|---|---|
| O que é ajustado | Estágios 6 e 7 + `top_conv` (blocos mais profundos) | Etapa 1: estágio 7 + topo → Etapa 2: + estágios 5 e 6 → Etapa 3: **base inteira** |
| Learning rate | 1e-4 fixo | 1e-4 → 3e-5 → 1e-5 (diminui à medida que camadas mais genéricas são liberadas) |
| Épocas (máx.) | 15 | 5 + 5 + 5 = 15 |
| Hipótese | As camadas profundas codificam padrões específicos do ImageNet (objetos); adaptá-las às texturas de lesões deve bastar, com menos risco de overfitting | Liberar camadas gradualmente, com lr decrescente, permite adaptar também filtros de nível médio (bordas, manchas, cores) sem "esquecer" o pré-treino de uma vez |

**O que permanece fixo (para que a diferença seja interpretável):** mesmos dados e divisão, mesma augmentation, mesmos pesos de classe, otimizador Adam, batch 32, perda, **BatchNormalization congelada** (`training=False`), o mesmo orçamento de 15 épocas, a mesma regra de seleção (maior F1 macro de validação com *early stopping*) e a mesma seed.

**O que muda:** **quantas** camadas são ajustadas, **quando** são liberadas e o *learning rate*.

**Por que lr menor que o da base congelada (1e-3):** pesos pré-treinados já são bons. Um passo grande destruiria o que foi aprendido no ImageNet (*catastrophic forgetting*).

**O que vamos registrar:** parâmetros treináveis, épocas, tempo, curvas, métricas com IC, matriz de confusão e as transições de erro nas imagens acompanhadas.

In [ ]:
STRATEGIES = {
    "A": [dict(prefixes=("block6", "block7", "top_"), lr=1e-4, epochs=15)],
    "B": [dict(prefixes=("block7", "top_"), lr=1e-4, epochs=5),
          dict(prefixes=("block5", "block6", "block7", "top_"), lr=3e-5, epochs=5),
          dict(prefixes=None, lr=1e-5, epochs=5)],
}

def set_trainable(base, prefixes):
    base.trainable = True
    for l in base.layers:
        chosen = prefixes is None or l.name.startswith(prefixes)
        l.trainable = bool(chosen) and not isinstance(l, layers.BatchNormalization)

def run_strategy(name):
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    model, base, head = build_model()
    model.load_weights(W_FROZEN)                # cópia do mesmo ponto de partida
    hist, times, marks, ep, params = {}, [], [], 0, []
    w_path = OUT_DIR / "modelos" / f"estrategia_{name}.weights.h5"
    for k, st in enumerate(STRATEGIES[name]):
        set_trainable(base, st["prefixes"])
        compile_model(model, st["lr"])
        params.append(n_trainable(model))
        print(f"\n[{name}] etapa {k + 1}: camadas {st['prefixes'] or 'TODAS'} | lr {st['lr']} | treináveis {params[-1]:,}")
        best = max(hist["val_f1_macro"]) if hist else None   # o arquivo guarda o melhor de TODAS as etapas
        h, t = fit(model, max(1, round(st["epochs"] * FT_EPOCH_SCALE)), w_path, initial_epoch=ep, best_so_far=best)
        for key, v in h.history.items():
            hist.setdefault(key, []).extend(v)
        times += t; ep += len(t); marks.append(ep)
    HISTS[f"estratégia {name}"] = hist; WEIGHT_PATHS[f"estratégia {name}"] = w_path
    model.load_weights(w_path)
    plot_curves(hist, f"estrategia {name}", marks=marks[:-1])
    m, prob = evaluate(model, VAL_DS, Y_VAL, f"estrategia {name}")
    RESULTS[f"estratégia {name}"] = {**m, "parâmetros treináveis": max(params), "épocas": ep, "tempo (min)": sum(times) / 60}
    model.save(OUT_DIR / "modelos" / f"estrategia_{name}.keras")
    del model, base, head; gc.collect()
    return prob

PROBS = {"base congelada": PROB_FROZEN}
HISTS = {"base congelada": H_FROZEN.history}
WEIGHT_PATHS = {"base congelada": W_FROZEN}
if RUN_FINE_TUNING:
    for s in STRATEGIES:
        PROBS[f"estratégia {s}"] = run_strategy(s)
    with open(OUT_DIR / "resultados_validacao.json", "w") as f:
        json.dump(RESULTS, f, indent=2, ensure_ascii=False, default=float)
    cols = ["F1 macro", "F1 macro IC95%", "balanced accuracy", "acurácia", "taxa de falso saudável",
            "parâmetros treináveis", "épocas", "tempo (min)"]
    display(pd.DataFrame(RESULTS).T[cols])
    p = {k: v.argmax(1) for k, v in PROBS.items()}
    for s in STRATEGIES:
        f0, f1 = p["base congelada"] == Y_VAL, p[f"estratégia {s}"] == Y_VAL
        print(f"Estratégia {s}: erros corrigidos {int((~f0 & f1).sum())}, persistentes {int((~f0 & ~f1).sum())}, "
              f"novos {int((f0 & ~f1).sum())}")
else:
    print("RUN_FINE_TUNING = False → estratégias A e B planejadas, ainda não executadas.")

---
## 8. Comparação das três condições (validação) — etapa 5 da lauda

Comparamos **base congelada**, **estratégia A** e **estratégia B** com os mesmos dados de validação, as mesmas métricas e a mesma regra de seleção.

**Como ler as diferenças:** o **IC 95% pareado** da diferença de F1 macro reamostra as **mesmas** imagens para os dois modelos. Se o intervalo inclui 0, a diferença não se distingue de ruído com ~400 imagens de validação.

In [ ]:
CONDS = ["base congelada", "estratégia A", "estratégia B"]
READY = RUN_FINE_TUNING and all(c in PROBS for c in CONDS)
if not READY:
    print("Comparação indisponível: rode as estratégias A e B (RUN_FINE_TUNING = True, Seção 7).")
else:
    PRED = {c: PROBS[c].argmax(1) for c in CONDS}
    CONF = {c: PROBS[c].max(1) for c in CONDS}
    cols = ["F1 macro", "F1 macro IC95%", "balanced accuracy", "acurácia", "taxa de falso saudável",
            "saudável prevista como doente", "parâmetros treináveis", "épocas", "tempo (min)"]
    display(Markdown("**Tabela comparativa — validação**"))
    display(pd.DataFrame(RESULTS).T.loc[CONDS, cols])

    def f1m(p, idx=slice(None)):
        return f1_score(Y_VAL[idx], p[idx], average="macro", labels=range(K), zero_division=0)
    rng = np.random.default_rng(SEED); boot = [rng.integers(0, len(Y_VAL), len(Y_VAL)) for _ in range(N_BOOT)]
    rows = []
    for a, b in [("base congelada", "estratégia A"), ("base congelada", "estratégia B"), ("estratégia A", "estratégia B")]:
        d = [f1m(PRED[b], i) - f1m(PRED[a], i) for i in boot]
        lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({"comparação": f"{b} − {a}", "Δ F1 macro": round(f1m(PRED[b]) - f1m(PRED[a]), 4),
                     "IC95% pareado": f"{lo:+.3f} a {hi:+.3f}",
                     "diferença real?": "sim" if (lo > 0 or hi < 0) else "não (IC inclui 0)"})
    display(Markdown("**Diferenças de F1 macro (bootstrap pareado):**"))
    display(pd.DataFrame(rows).set_index("comparação"))

    pc = pd.DataFrame({c: f1_score(Y_VAL, PRED[c], average=None, labels=range(K), zero_division=0) for c in CONDS},
                      index=CLASSES).round(3)
    pc["Δ A"] = (pc["estratégia A"] - pc["base congelada"]).round(3)
    pc["Δ B"] = (pc["estratégia B"] - pc["base congelada"]).round(3)
    pc["n validação"] = pd.Series(Y_VAL).value_counts().reindex(range(K)).values
    display(Markdown("**F1 por classe — quais classes melhoraram ou pioraram:**"))
    display(pc)

    nF = len(HISTS["base congelada"]["val_f1_macro"])
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    for key, a, lab in [("val_f1_macro", ax[0], "F1 macro (validação)"), ("val_loss", ax[1], "loss (validação)")]:
        H = HISTS["base congelada"][key]
        a.plot(range(1, nF + 1), H, "o-", c="gray", label="base congelada")
        for c, col in [("estratégia A", "#1f77b4"), ("estratégia B", "#d62728")]:
            Hc = HISTS[c][key]
            a.plot(range(nF + 1, nF + len(Hc) + 1), Hc, "s-", c=col, label=c)
        a.axvline(nF + 0.5, ls="--", c="black"); a.set_xlabel("época"); a.set_title(lab); a.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "curvas_tres_condicoes.png", dpi=110); plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(20, 5.6))
    for ax_, c in zip(axes, CONDS):
        cm = confusion_matrix(Y_VAL, PRED[c], labels=range(K), normalize="true")
        ax_.imshow(cm, cmap="Blues", vmin=0, vmax=1)
        for i in range(K):
            for j in range(K):
                ax_.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cm[i, j] > 0.5 else "black")
        ax_.set_xticks(range(K)); ax_.set_xticklabels(CLASSES, rotation=40, ha="right", fontsize=8)
        ax_.set_yticks(range(K)); ax_.set_yticklabels(CLASSES, fontsize=8)
        ax_.set_title(f"{c} — F1 macro {RESULTS[c]['F1 macro']:.3f}"); ax_.set_xlabel("previsto"); ax_.set_ylabel("real")
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "matrizes_tres_condicoes.png", dpi=110); plt.show()

**📝 Discussão**

- **Houve ganho com o fine-tuning, e ele é real.** O F1 macro subiu de 0,903 (base congelada) para **0,950 na estratégia A** e **0,952 na B**. Os ICs pareados das diferenças ficam inteiramente acima de zero (A: +0,016 a +0,080; B: +0,022 a +0,077). A acurácia foi de 0,920 para 0,961 nas duas, e a *balanced accuracy* de 0,914 para 0,946 (A) e 0,952 (B).
- **A e B empatam.** A diferença entre elas é de +0,002 em F1 macro, com IC pareado de −0,015 a +0,018: não se distingue de ruído com 388 imagens.
- **Classes que melhoraram:** ferrugem (F1 0,951 → 0,993 em A / 0,989 em B), saudável (0,895 → 0,971 nas duas, porque a precisão subiu de 0,829 para 0,971), mancha de phoma (0,933 → 0,965 / 0,958), cercosporiose (0,788 → 0,867 / 0,875) e bicho-mineiro (0,947 → 0,955 / 0,966). **Nenhuma classe piorou.**
- **Erro mais grave:** o falso saudável caiu de **7 casos (1,98%) para 1 (0,28%)** nas duas estratégias. As 5 ferrugens classificadas como saudáveis foram todas corrigidas.
- **A cercosporiose continua sendo a classe mais difícil** (F1 ~0,87). Na estratégia A, ela perde 6 imagens para bicho-mineiro; na B, 3.
- **Overfitting:** na A, o F1 de treino chegou a 0,99, enquanto o de validação teve pico na época 6 (0,950) e depois oscilou entre 0,92 e 0,95, com a loss de validação subindo de 0,121 (época 3) para 0,207 (época 10). Há **sinal de overfitting**, contido pelo early stopping e pela restauração dos melhores pesos. Na B, o F1 de validação subiu de forma mais estável até a última época (0,952) e a loss de validação ficou em torno de 0,15: o descongelamento gradual com lr decrescente deixou o treino **mais estável**, embora o F1 de treino também tenha chegado a ~0,99.
- **O ganho justificou o custo?** Em relação à base congelada, **sim**: com 7,2 min (A) ou 12,7 min (B) de treino a mais, os erros na validação caíram pela metade (31 → 15 imagens) e o falso saudável quase desapareceu. Já **B não compensa em relação a A**: custa 76% mais tempo e ajusta 22% mais parâmetros (6,46 M × 5,27 M) para uma diferença que não se distingue de ruído.

## 9. Análise visual dos erros nas três condições

Acompanhamos as **mesmas imagens** nos três modelos:
- **Erro corrigido:** a base congelada errou e o fine-tuning acertou.
- **Erro persistente:** as três condições erraram.
- **Erro novo:** a base congelada acertou e o fine-tuning errou.

Cada imagem mostra a classe real e, para cada condição, a classe prevista e o score (verde = acerto, vermelho = erro).

In [ ]:
SHORT = {"base congelada": "Congelada", "estratégia A": "FT-A", "estratégia B": "FT-B"}

def show3(idx, title, ncols=4):
    idx = list(idx)
    if not idx:
        print(f"{title}: nenhum caso."); return
    nrows = int(np.ceil(len(idx) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 3.9 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for a in axes: a.axis("off")
    for a, i in zip(axes, idx):
        a.imshow(Image.open(VAL.path[i]).convert("RGB"))
        a.set_title(f"real: {CLASSES[Y_VAL[i]]}", fontsize=9, weight="bold")
        for k, c in enumerate(CONDS):
            p = PRED[c][i]
            a.text(0.0, -0.05 - 0.1 * k, f"{SHORT[c]}: {CLASSES[p]} ({CONF[c][i]:.2f})", transform=a.transAxes,
                   fontsize=8, va="top", color="green" if p == Y_VAL[i] else "red")
    fig.suptitle(title, fontsize=11); fig.subplots_adjust(hspace=0.75, top=0.9); plt.show()

if READY:
    OK = {c: PRED[c] == Y_VAL for c in CONDS}
    trans = {}
    for s in ["estratégia A", "estratégia B"]:
        f0, f1 = OK["base congelada"], OK[s]
        trans[s] = {"acertos mantidos": int((f0 & f1).sum()), "erros corrigidos": int((~f0 & f1).sum()),
                    "erros persistentes": int((~f0 & ~f1).sum()), "erros novos": int((f0 & ~f1).sum())}
    display(Markdown("**Transições em relação à base congelada (validação inteira):**"))
    display(pd.DataFrame(trans))
    for c in CONDS:
        if OK[c].all():
            print(f"{c}: nenhum erro na validação — analisar acertos e os casos mais incertos.")

    CORRECTED = np.where(~OK["base congelada"] & OK["estratégia A"] & OK["estratégia B"])[0]
    PERSIST = np.where(~OK["base congelada"] & ~OK["estratégia A"] & ~OK["estratégia B"])[0]
    NEW = np.where(OK["base congelada"] & (~OK["estratégia A"] | ~OK["estratégia B"]))[0]
    show3([i for i in TRACKED][:12], "Imagens acompanhadas desde a análise preliminar (Seção 6)")
    show3(CORRECTED[:8], f"Erros corrigidos por A e B ({len(CORRECTED)} no total)")
    show3(PERSIST[:8], f"Erros persistentes nas três condições ({len(PERSIST)} no total)")
    show3(NEW[:8], f"Erros novos após o fine-tuning ({len(NEW)} no total)")
    if len(PERSIST):
        display(Markdown("**Confusões dos erros persistentes (real → previsto pela estratégia A):**"))
        display(pd.Series([f"{CLASSES[Y_VAL[i]]} → {CLASSES[PRED['estratégia A'][i]]}" for i in PERSIST])
                .value_counts().rename("casos").to_frame())
    display(Markdown("**Taxa de erro por fonte das imagens:**"))
    display(pd.DataFrame({"fonte": VAL.source, **{c: ~OK[c] for c in CONDS}}).groupby("fonte").mean().round(3))

**📝 Hipóteses**

- **Transições:** a estratégia A corrigiu 22 erros da base congelada, manteve 9 e criou 6 novos; a B corrigiu 18, manteve 13 e criou só 2. As duas terminam com 15 erros, mas por caminhos diferentes: a A é mais "agressiva" (corrige mais e cria mais erros novos), a B é mais conservadora.
- **Erros corrigidos:** incluem as 5 ferrugens que a base congelada chamava de saudáveis. A hipótese é que sejam lesões iniciais ou pequenas, cujas pústulas só passaram a ser reconhecidas quando as camadas profundas foram ajustadas. O Grad-CAM da Seção 10 mostra isso num exemplo.
- **Erros persistentes (9 casos, os mesmos nas três condições):** 7 de 9 envolvem **cercosporiose**, como classe real ou prevista (confundida com phoma ou bicho-mineiro). Dois casos são confiantes demais para serem só "dificuldade do modelo": uma cercosporiose prevista como phoma com score 1,00 e uma ferrugem prevista como cercosporiose com 0,99. Como os três modelos concordam com alta confiança, a hipótese mais forte é **ambiguidade de sintoma ou de rótulo** (lesão com características das duas doenças, ou folha com mais de um sintoma: 101 folhas têm recortes de classes diferentes). Também persistem uma folha saudável prevista como ferrugem (0,95) e uma phoma do Digipathos prevista como saudável, o único falso saudável que restou.
- **Erros novos (6):** 5 dos 6 são imagens de **cercosporiose**, e 4 delas passaram a ser classificadas como **bicho-mineiro** na estratégia A. Ao especializar as camadas profundas, a A deslocou a fronteira entre essas duas classes; a B, que ajusta mais devagar, criou só 2 erros novos.
- **Fonte das imagens:** a taxa de erro caiu nas duas fontes. Nas imagens dos autores, de 7,2% para 3,8% (A) e 3,4% (B); nas do Digipathos, de 10,4% para 4,2% e 5,2%. A diferença entre as fontes praticamente desapareceu, o que **enfraquece a hipótese de atalho de fonte** levantada na Seção 3, pelo menos na validação.

## 10. Grad-CAM nas três condições — etapa 6 da lauda

- **Método:** Grad-CAM, adequado a CNNs com a cabeça GAP + Dense.
- **Camada investigada:** `top_activation`, a saída da última convolução da EfficientNetB1 (mapa 8×8).
- **Classe investigada:** a classe **prevista** por cada condição (indicada no título).

As imagens escolhidas incluem erros corrigidos, erros persistentes, erros novos e um acerto mantido, sempre as **mesmas** nas três condições.

In [ ]:
def gradcam_with(base, head, x_uint8, class_idx):
    W, b = [tf.constant(w) for w in head.get_layer("pred").get_weights()]
    x = tf.cast(x_uint8[None], tf.float32)
    with tf.GradientTape() as tape:
        fmap = base(PREPROCESS(x), training=False)
        tape.watch(fmap)
        score = (tf.matmul(tf.reduce_mean(fmap, axis=(1, 2)), W) + b)[:, class_idx]
    g = tape.gradient(score, fmap)[0]
    cam = tf.nn.relu(tf.reduce_sum(fmap[0] * tf.reduce_mean(g, axis=(0, 1)), -1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return tf.image.resize(cam[..., None].astype(np.float32), (IMG_SIZE, IMG_SIZE)).numpy()[..., 0]

if READY:
    nets = {}
    for c in CONDS:
        m_, b_, h_ = build_model(); m_.load_weights(WEIGHT_PATHS[c]); nets[c] = (b_, h_)
    MAINTAINED = np.where(OK["base congelada"] & OK["estratégia A"] & OK["estratégia B"])[0]
    pick = list(CORRECTED[:2]) + list(PERSIST[:2]) + list(NEW[:1]) + list(MAINTAINED[:1])
    for pool in (CORRECTED, PERSIST, MAINTAINED):
        for i in pool:
            if len(pick) >= 6: break
            if i not in pick: pick.append(i)
    CAM3_IDX = [int(i) for i in pick[:6]]
    fig, axes = plt.subplots(len(CAM3_IDX), 4, figsize=(13, 3.4 * len(CAM3_IDX)))
    axes = np.atleast_2d(axes)
    for r, i in enumerate(CAM3_IDX):
        img = load_img(VAL.path[i]).numpy()
        status = " ".join(f"{SHORT[c]}{'✓' if OK[c][i] else '✗'}" for c in CONDS)
        axes[r, 0].imshow(img); axes[r, 0].set_title(f"real: {CLASSES[Y_VAL[i]]}\n{status}", fontsize=8)
        for k, c in enumerate(CONDS, start=1):
            p = int(PRED[c][i]); cam = gradcam_with(*nets[c], img, p)
            axes[r, k].imshow(np.clip(0.55 * img / 255 + 0.45 * plt.cm.jet(cam)[..., :3], 0, 1))
            axes[r, k].set_title(f"{SHORT[c]}: {CLASSES[p]} ({CONF[c][i]:.2f})\nclasse investigada: {CLASSES[p]}",
                                 fontsize=8, color="green" if p == Y_VAL[i] else "red")
        for a in axes[r]: a.axis("off")
    fig.suptitle(f"Grad-CAM nas três condições — camada '{CAM_LAYER}' (EfficientNetB1), validação", fontsize=11)
    plt.tight_layout(rect=(0, 0, 1, 0.96)); plt.savefig(OUT_DIR / "figs" / "gradcam_tres_condicoes.png", dpi=110); plt.show()
    del nets; gc.collect()

**📝 Análise dos mapas** (camada `top_activation`, classe prevista)

- **Falso saudável corrigido (2ª linha):** na base congelada, o mapa destaca o **tecido verde ao redor** da pústula e deixa a pústula "fria" (azul); o modelo previu saudável (0,50). Depois do fine-tuning, A e B **concentram o mapa exatamente sobre a pústula alaranjada** e preveem ferrugem com score 1,00. É o exemplo mais claro de que o fine-tuning ensinou o modelo a olhar para a lesão.
- **Bicho-mineiro corrigido (1ª linha, imagem do Digipathos):** a base congelada espalha o destaque pela lesão e por parte do fundo e prevê cercosporiose (0,62); A e B concentram o mapa na lesão e acertam com 0,99–1,00.
- **Erros persistentes (3ª e 4ª linhas):** nas três condições, o mapa cai **sobre a lesão**, não no fundo. Os modelos olham para o lugar certo e mesmo assim erram com alta confiança (ferrugem → cercosporiose; cercosporiose → phoma). Isso reforça a hipótese de **ambiguidade visual ou de rótulo**, e não de atalho.
- **Erro novo da estratégia A (5ª linha):** um bicho-mineiro que a base congelada e a B acertam, mas que a A classifica como phoma (0,76). O mapa da A continua sobre a lesão: a mudança está na interpretação da textura, não no lugar observado, o que é coerente com o deslocamento de fronteira visto na Seção 9.
- **Folha saudável (6ª linha):** sem lesão, o mapa se espalha por toda a superfície da folha, o que é esperado para uma classe definida pela **ausência** de sintomas. Não há destaque concentrado na borda nem no fundo.
- **Atalho de fonte:** nos exemplos analisados, não vimos destaque sistemático nas bordas ou no fundo das imagens do Digipathos. Junto com a queda da diferença de erro entre as fontes (Seção 9), isso sugere que o atalho levantado na Seção 3 não domina o modelo após o fine-tuning, mas 6 imagens não bastam para descartá-lo.

---
## 11. Escolha do modelo recomendado (só pela validação) — etapa 7 da lauda

**Regra definida antes de olhar o teste:**
1. Escolher a condição de **maior F1 macro na validação**.
2. Se a diferença para outra condição for **menor que 0,01** (dentro do ruído), desempatar pela **menor taxa de falso saudável** (o erro mais grave) e depois pelo **menor custo** (parâmetros treináveis).

In [ ]:
CHOSEN = None
if READY:
    f1s = {c: RESULTS[c]["F1 macro"] for c in CONDS}
    top = max(f1s.values())
    tied = [c for c in CONDS if top - f1s[c] < 0.01]
    CHOSEN = min(tied, key=lambda c: (RESULTS[c]["taxa de falso saudável"], RESULTS[c]["parâmetros treináveis"]))
    why = ("maior F1 macro de validação" if len(tied) == 1 else
           f"empate técnico entre {tied} (diferença < 0,01): desempate pela menor taxa de falso saudável e depois pelo menor custo")
    display(Markdown(f"### Modelo recomendado: **{CHOSEN}**\n\nCritério: {why}. "
                     f"F1 macro de validação = {f1s[CHOSEN]:.3f}; taxa de falso saudável = "
                     f"{RESULTS[CHOSEN]['taxa de falso saudável']:.3f}; parâmetros treináveis = {RESULTS[CHOSEN]['parâmetros treináveis']:,}."))
    with open(OUT_DIR / "escolha_do_modelo.json", "w") as f:
        json.dump({"escolhido": CHOSEN, "criterio": why, "f1_validacao": f1s,
                   "registrado_em": time.strftime("%Y-%m-%d %H:%M:%S")}, f, indent=2, ensure_ascii=False)
else:
    print("Escolha indisponível: rode antes as estratégias A e B.")

**📝 Justificativa**

- A regra apontou **empate técnico** entre A (F1 0,950) e B (0,952): diferença menor que 0,01 e dentro do ruído (IC pareado −0,015 a +0,018).
- O primeiro desempate, a taxa de falso saudável, também empatou: 0,28% nas duas (1 caso).
- Pelo segundo critério, **menor custo**, venceu a **estratégia A**: 5,27 M parâmetros treináveis contra 6,46 M, e 7,2 min de treino contra 12,7 min.
- **Para a finalidade** (aplicativo de campo), a A também faz sentido: recall de 0,993 na ferrugem e de 0,971 na classe saudável, falso saudável de 0,28% e mapas Grad-CAM concentrados nas lesões.
- **Ponto de atenção:** a A mostrou mais sinais de overfitting e criou mais erros novos que a B (6 × 2, quase todos cercosporiose → bicho-mineiro). Se o teste ficar abaixo do esperado, essa é a primeira hipótese a investigar, **sem trocar o modelo por causa do teste**.

## 12. Avaliação final no conjunto de teste (uma única vez)

Só agora, **depois de registrada a escolha** (arquivo `escolha_do_modelo.json`), o teste é liberado. Os três modelos são avaliados no **mesmo** conjunto de teste reservado desde a Seção 3.6.

**O teste não muda a escolha nem orienta novos ajustes:** serve apenas para verificar se o desempenho esperado pela validação se confirma.

> Para liberar, defina `RUN_FINAL_TEST = True` na configuração. Fica desligado por padrão para o teste não ser usado por acidente.

In [ ]:
TEST_RESULTS = {}
if READY and CHOSEN and RUN_FINAL_TEST:
    TEST_UNLOCKED = True
    TEST = get_split("test")
    TEST_DS = make_ds(TEST, False)
    Y_TEST = TEST.label.map(CLASS_TO_ID).values
    print(f"Conjunto de teste: {len(TEST)} imagens (folhas que nunca apareceram no treino nem na validação)")
    for c in CONDS:
        m_, _, _ = build_model(); m_.load_weights(WEIGHT_PATHS[c])
        TEST_RESULTS[c], _ = evaluate(m_, TEST_DS, Y_TEST, f"{c} (teste)", split_name="TESTE")
        del m_; gc.collect()
    comp = pd.DataFrame({c: {"F1 macro — validação": RESULTS[c]["F1 macro"], "F1 macro — teste": TEST_RESULTS[c]["F1 macro"],
                             "diferença (teste − val)": TEST_RESULTS[c]["F1 macro"] - RESULTS[c]["F1 macro"],
                             "F1 macro teste IC95%": str(TEST_RESULTS[c]["F1 macro IC95%"]),
                             "balanced acc — validação": RESULTS[c]["balanced accuracy"],
                             "balanced acc — teste": TEST_RESULTS[c]["balanced accuracy"],
                             "falso saudável — teste": TEST_RESULTS[c]["taxa de falso saudável"]} for c in CONDS}).T
    display(Markdown(f"**Validação × teste** (modelo recomendado: **{CHOSEN}**)"))
    display(comp.round(4))
    with open(OUT_DIR / "resultados_teste.json", "w") as f:
        json.dump({"escolhido_antes_do_teste": CHOSEN, "teste": TEST_RESULTS}, f, indent=2, ensure_ascii=False, default=float)
elif READY:
    print("Teste ainda reservado. Defina RUN_FINAL_TEST = True somente depois de registrar a escolha (Seção 11).")
else:
    print("Teste indisponível: rode antes as estratégias A e B.")

**📝 Discussão (preencher):** os resultados do teste foram compatíveis com a validação (diferença dentro dos ICs)? A ordem entre as condições se manteve? Se o teste ficou bem abaixo da validação, quais hipóteses explicam (validação pequena, seleção otimista pela validação, folhas diferentes)? **Lembrete:** a escolha não muda por causa do teste.

## 13. Salvamento, carregamento e uso do modelo escolhido

1. O modelo recomendado é salvo como `modelo_recomendado.keras`, junto com `modelo_recomendado_info.json`, que guarda as classes **na ordem dos índices**, o tamanho de entrada, o pré-processamento e as métricas.
2. O modelo é **recarregado do disco, sem treinar**, e a previsão é conferida contra a do modelo em memória: tem que dar o mesmo resultado, o que prova que o pré-processamento e o mapa índice → classe foram preservados.
3. **Imagem externa da internet:** escolham uma foto de uma das 5 classes e preencham `EXTERNAL_IMAGE_URL` (link direto para a imagem, ou caminho de um arquivo enviado ao Colab), `EXTERNAL_IMAGE_SOURCE` (página de origem e licença) e `EXTERNAL_EXPECTED` (a classe esperada).
   - Uma boa fonte é o **Wikimedia Commons** (ex.: busquem "coffee leaf rust" ou "Hemileia vastatrix"), que informa a licença de cada imagem.

> Essa demonstração mostra a reutilização do modelo. Ela **não substitui** a avaliação no teste e **não exige acerto**: a foto externa costuma ter enquadramento diferente dos recortes do dataset (folha inteira, fundo, iluminação), e isso é um bom ponto de discussão.

In [ ]:
FILES = {"base congelada": "base_congelada.keras", "estratégia A": "estrategia_A.keras", "estratégia B": "estrategia_B.keras"}
if CHOSEN:
    FINAL = OUT_DIR / "modelos" / "modelo_recomendado.keras"
    shutil.copy(OUT_DIR / "modelos" / FILES[CHOSEN], FINAL)
    INFO = {"modelo": f"EfficientNetB1 — {CHOSEN}", "classes": CLASSES, "indice_para_classe": {i: c for i, c in enumerate(CLASSES)},
            "img_size": IMG_SIZE, "entrada": "imagem RGB com pixels 0-255, redimensionada para 240x240 (bilinear, sem preservar proporção); "
                                             "a normalização ImageNet está dentro do modelo",
            "validacao": RESULTS[CHOSEN], "teste": TEST_RESULTS.get(CHOSEN), "versoes": VERSIONS}
    with open(OUT_DIR / "modelos" / "modelo_recomendado_info.json", "w") as f:
        json.dump(INFO, f, indent=2, ensure_ascii=False, default=float)
    print("Salvo:", FINAL)

    # --- recarregar do disco, como faria um usuário do modelo (sem treinar) ---
    LOADED = keras.models.load_model(FINAL)
    with open(OUT_DIR / "modelos" / "modelo_recomendado_info.json") as f:
        INFO_LOADED = json.load(f)
    IDX2CLS = {int(k): v for k, v in INFO_LOADED["indice_para_classe"].items()}

    def predict_image(path):
        img = load_img(str(path)).numpy()          # mesmo redimensionamento usado no treino
        prob = LOADED.predict(tf.cast(img[None], tf.float32), verbose=0)[0]
        return img, prob

    _, p_check = predict_image(VAL.path[0])
    same = np.allclose(p_check, PROBS[CHOSEN][0], atol=1e-4)
    print(f"Conferência: previsão do modelo recarregado = previsão original? {same} "
          f"(classe {IDX2CLS[int(p_check.argmax())]}, score {p_check.max():.4f})")
else:
    print("Salvamento indisponível: escolha o modelo antes (Seção 11).")

In [ ]:
EXTERNAL_IMAGE_URL = "https://raw.githubusercontent.com/VithorDosSantos/Cnn_Coffe/main/imagem_externa/ferrugem_hemileia_vastatrix_carvalho2011.png"  # link direto para a imagem (.jpg/.png) OU caminho de arquivo enviado ao Colab
EXTERNAL_IMAGE_SOURCE = "Wikimedia Commons — File:Hemileia_vastatrix_Leaf_symptoms.png; Carvalho et al. (2011), PLoS ONE, CC BY 2.5 (cópia no GitHub do grupo)"  # página de origem + autor/licença (ex.: página do Wikimedia Commons)
EXTERNAL_EXPECTED = "ferrugem"  # classe esperada: saudavel, bicho_mineiro, ferrugem, mancha_phoma ou cercosporiose

if CHOSEN and EXTERNAL_IMAGE_URL:
    dst = DATA_DIR / "imagem_externa.jpg"
    if Path(EXTERNAL_IMAGE_URL).exists():
        raw = Path(EXTERNAL_IMAGE_URL).read_bytes()
    else:
        r = requests.get(EXTERNAL_IMAGE_URL, timeout=60, headers={"User-Agent": "cnn-coffee-coursework/1.0 (notebook academico; python-requests)"})
        r.raise_for_status(); raw = r.content
    Image.open(io.BytesIO(raw)).convert("RGB").save(dst, quality=95)   # aceita webp/png/jpg
    img, prob = predict_image(dst)
    top = np.argsort(-prob)[:3]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [1, 1.2]})
    ax[0].imshow(Image.open(dst)); ax[0].axis("off")
    ax[0].set_title(f"esperada: {EXTERNAL_EXPECTED or '?'} | prevista: {IDX2CLS[int(top[0])]} ({prob[top[0]]:.2f})", fontsize=9,
                    color="green" if IDX2CLS[int(top[0])] == EXTERNAL_EXPECTED else "red")
    ax[1].barh([IDX2CLS[i] for i in range(K)][::-1], prob[::-1], color="#4C72B0"); ax[1].set_xlim(0, 1)
    ax[1].set_title("probabilidade por classe (modelo recarregado)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "imagem_externa.png", dpi=110); plt.show()
    display(pd.Series({"fonte": EXTERNAL_IMAGE_SOURCE or "(preencher)", "classe esperada": EXTERNAL_EXPECTED or "(preencher)",
                       "previsão": IDX2CLS[int(top[0])], "score": round(float(prob[top[0]]), 4),
                       "2ª opção": f"{IDX2CLS[int(top[1])]} ({prob[top[1]]:.2f})",
                       "3ª opção": f"{IDX2CLS[int(top[2])]} ({prob[top[2]]:.2f})"}, name="imagem externa").to_frame())
elif CHOSEN:
    print("Preencha EXTERNAL_IMAGE_URL, EXTERNAL_IMAGE_SOURCE e EXTERNAL_EXPECTED e rode esta célula.")

**📝 Comentário sobre a imagem externa**

- **Imagem:** face inferior de uma folha de café **inteira** com pústulas de ferrugem (Carvalho et al., 2011; Wikimedia Commons, CC BY 2.5). Classe esperada: **ferrugem**.
- **Resultado:** o modelo recomendado, **recarregado do disco sem treino**, previu **ferrugem com score 0,9988**; as outras classes ficaram próximas de zero.
- **Por que é um bom teste:** a foto é bem diferente dos recortes do dataset. Mostra a folha inteira, com fundo de vegetação, várias pústulas espalhadas, uma área necrótica marrom (que lembra cercosporiose ou phoma) e uma barra de escala. Mesmo assim, o modelo acertou com alta confiança, porque as pústulas alaranjadas da ferrugem são muito características.
- **Limite:** é **uma** imagem, de uma classe visualmente marcante; isso **não** mede o desempenho em campo. Folhas inteiras com várias doenças, outras iluminações ou fotos de celular podem confundir o modelo (*domain shift*). Um aplicativo real precisaria de uma etapa que localize e recorte as lesões antes da classificação.
- A conferência da célula anterior mostrou que o modelo recarregado dá **exatamente a mesma previsão** que o modelo em memória: o pré-processamento e o mapa índice → classe foram preservados.

## 14. Recomendação final

**Recomendamos a EfficientNetB1 com a estratégia A** (fine-tuning dos estágios 6 e 7 e da convolução final, lr 1e-4, BatchNorm congelada). Na validação, ela elevou o F1 macro de 0,903 para **0,950** (IC95% 0,926–0,973) e a *balanced accuracy* para 0,946, e reduziu o falso saudável de 1,98% para **0,28%**, com 7,2 min de treino na T4. A estratégia B chegou a um resultado equivalente (0,952), mas com 76% mais tempo de treino e 22% mais parâmetros ajustados, sem ganho distinguível de ruído.

**Resultado no teste:** _(preencher depois de rodar a Seção 12)_.

**Limitações:**
- validação e teste pequenos (~400 imagens cada; 35 a 48 saudáveis), com intervalos de confiança largos;
- mistura de fontes (autores × Digipathos), com possíveis atalhos de resolução;
- a cercosporiose ainda é a classe mais difícil (F1 ~0,87), com erros persistentes que sugerem ambiguidade de rótulo;
- o modelo foi treinado com **recortes** de lesões, não com fotos inteiras de campo.

**Com mais tempo:** revisar com um especialista os erros persistentes de cercosporiose; repetir a divisão com várias seeds para estimar a variância dos resultados; avaliar separadamente as imagens do Digipathos; e coletar fotos de campo, de celular, para medir o *domain shift*.